# Construire et valider la couche Silver

## Objectif concret

Ce notebook retrace le passage réellement implémenté de Bronze vers Silver pour Indusense. Il permet de comprendre le modèle cible, les types, les relations, la traçabilité, les transformations, la migration Alembic et les contrôles effectués après publication dans PostgreSQL.

**Point de départ observable :** les sources sont déjà ingérées dans Bronze et le pipeline Silver a déjà été exécuté. Toutes les cellules de ce notebook sont en lecture seule : un **Run All** ne remplace aucune table.

In [1]:
from pathlib import Path
import os

from tabulate import tabulate

# Le notebook doit être ouvert depuis la racine du projet Indusense.
PROJECT_ROOT = Path.cwd()
assert (PROJECT_ROOT / "pyproject.toml").exists(), (
    "Ouvrir le notebook depuis le dossier racine indusense."
)

# Les paramètres PostgreSQL sont chargés sans afficher leurs valeurs sensibles.
ENV_FILE = PROJECT_ROOT / ".docker" / ".env"
for raw_line in ENV_FILE.read_text(encoding="utf-8").splitlines():
    line = raw_line.strip()
    if line and not line.startswith("#") and "=" in line:
        key, value = line.split("=", 1)
        if key.startswith("DB_"):
            os.environ.setdefault(key, value)

# L'environnement Compose utilise les valeurs PostgreSQL par défaut si hôte et port sont absents du fichier.
os.environ.setdefault("DB_HOST", "127.0.0.1")
os.environ.setdefault("DB_PORT", "5432")
required_variables = {"DB_USER", "DB_PASSWORD", "DB_NAME", "DB_HOST", "DB_PORT"}
assert required_variables <= os.environ.keys(), "Configuration PostgreSQL incomplète."

print(f"Projet : {PROJECT_ROOT.name}")
print("Configuration PostgreSQL chargée sans exposer les secrets.")

Projet : indusense
Configuration PostgreSQL chargée sans exposer les secrets.


## 1. Définir le grain et les tables Silver

Le **grain** décrit ce que représente exactement une ligne. Silver sépare les quatre grains métier trouvés dans Bronze : une machine, un incident, une maintenance et une mesure de télémétrie.

La cellule suivante inspecte directement les modèles SQLAlchemy afin de vérifier les tables et leurs colonnes. Un **ORM** (*Object-Relational Mapper*) associe ici chaque classe Python à une table PostgreSQL.

In [2]:
from indusense.db.models import Incident, Machine, Maintenance, Telemetry

silver_models = [Machine, Incident, Maintenance, Telemetry]
model_summary = []

for model in silver_models:
    table = model.__table__
    model_summary.append(
        [
            f"{table.schema}.{table.name}",
            len(table.columns),
            ", ".join(column.name for column in table.primary_key.columns),
        ]
    )

print(tabulate(model_summary, headers=["Table", "Colonnes", "Clé primaire"], tablefmt="github"))
assert {model.__table__.name for model in silver_models} == {
    "machine", "incident", "maintenance", "telemetry"
}
print("\nLes quatre grains Silver sont déclarés.")

| Table              |   Colonnes | Clé primaire   |
|--------------------|------------|----------------|
| silver.machine     |         15 | machine_code   |
| silver.incident    |         20 | incident_id    |
| silver.maintenance |         17 | maintenance_id |
| silver.telemetry   |         14 | telemetry_id   |

Les quatre grains Silver sont déclarés.


## 2. Vérifier les clés et les relations

Une **clé primaire** identifie une ligne de façon unique. Une **clé étrangère** relie une ligne à une autre table et empêche les références orphelines. La télémétrie possède une clé technique auto-incrémentée, mais son unicité métier porte sur `(machine_code, measured_at)`.

Pour une maintenance réactive, la relation vers l'incident utilise simultanément `related_incident_id` et `machine_code`. Cette clé étrangère composite garantit que la maintenance et l'incident concernent la même machine après réalignement. La cellule suivante vérifie ces contraintes dans les modèles.

In [3]:
from sqlalchemy import ForeignKeyConstraint, UniqueConstraint

relation_summary = []
for model in silver_models:
    table = model.__table__
    foreign_keys = sorted(
        f"{foreign_key.parent.name} -> {foreign_key.target_fullname}"
        for foreign_key in table.foreign_keys
    )
    relation_summary.append([f"silver.{table.name}", " ; ".join(foreign_keys) or "aucune"])

print(tabulate(relation_summary, headers=["Table", "Clés étrangères"], tablefmt="github"))

maintenance_composite_fk = any(
    isinstance(constraint, ForeignKeyConstraint)
    and {element.parent.name for element in constraint.elements}
    == {"related_incident_id", "machine_code"}
    for constraint in Maintenance.__table__.constraints
)
telemetry_business_key = any(
    isinstance(constraint, UniqueConstraint)
    and {column.name for column in constraint.columns} == {"machine_code", "measured_at"}
    for constraint in Telemetry.__table__.constraints
)
assert maintenance_composite_fk
assert telemetry_business_key
print("\nRelations métier et unicité télémétrique vérifiées.")

| Table              | Clés étrangères                                                                                                                                                                                                                                |
|--------------------|------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| silver.machine     | pipeline_run_id -> ops.pipeline_run.run_id ; source_batch_id -> ops.ingestion_batch.batch_id                                                                                                                                                   |
| silver.incident    | machine_code -> silver.machine.machine_code ; pipeline_run_id -> ops.pipeline_run.run_id ; source_batch_id -> ops.ingestion_batch.batch_id                                               

## 3. Appliquer les règles de transformation

Silver transforme les chaînes Bronze en types métier : dates UTC, booléens, entiers, nombres décimaux et mesures numériques. La **minimisation** consiste à ne pas recopier les données inutiles au besoin : `operator_name`, `operator_badge` et `shift` disparaissent de l'incident Silver. Le texte libre `comment` est conservé, car il peut décrire un arrêt de machine qui n'est pas porté par un indicateur structuré.

Les maintenances réactives reprennent la machine de leur incident lié. Les télémétries partageant une machine et un timestamp sont des doublons techniques : la ligne la plus complète est conservée, puis la première ligne source en cas d'égalité. Une mesure de capteur absente reste `NULL` et génère un avertissement ; elle n'est jamais remplacée arbitrairement par zéro.

In [4]:
from indusense.ingestion.silver import PIPELINE_VERSION, TRANSFORMATION_VERSION

transformation_contract = [
    ["machine", "normaliser le code et typer capacités, date, criticité et activité"],
    ["incident", "fusionner date et heure en UTC, typer les indicateurs, conserver le commentaire métier et minimiser les données opérateur"],
    ["maintenance", "aligner la machine sur l'incident pour le réactif et conserver le code source"],
    ["telemetry", "typer les mesures, conserver les absences en NULL et dédupliquer par machine/timestamp"],
]
print(tabulate(transformation_contract, headers=["Grain", "Transformation principale"], tablefmt="github"))

incident_columns = set(Incident.__table__.columns.keys())
removed_columns = {"operator_name", "operator_badge", "shift"}
assert incident_columns.isdisjoint(removed_columns)
assert Incident.__table__.c.comment.type.python_type is str
assert Telemetry.__table__.c.temperature_c.nullable
assert Telemetry.__table__.c.pressure_bar.nullable
assert Telemetry.__table__.c.voltage_mean_v.nullable
assert Telemetry.__table__.c.rotation_mean_rpm.nullable

print(f"\nPipeline {PIPELINE_VERSION} — transformation {TRANSFORMATION_VERSION}")
print("Commentaire métier, minimisation et nullabilité des capteurs vérifiés.")

| Grain       | Transformation principale                                                                                                 |
|-------------|---------------------------------------------------------------------------------------------------------------------------|
| machine     | normaliser le code et typer capacités, date, criticité et activité                                                        |
| incident    | fusionner date et heure en UTC, typer les indicateurs, conserver le commentaire métier et minimiser les données opérateur |
| maintenance | aligner la machine sur l'incident pour le réactif et conserver le code source                                             |
| telemetry   | typer les mesures, conserver les absences en NULL et dédupliquer par machine/timestamp                                    |

Pipeline 1.1.0 — transformation 20260901_02
Commentaire métier, minimisation et nullabilité des capteurs vérifiés.


## 4. Versionner puis appliquer le schéma Silver

Une **migration Alembic** est un fichier versionné décrivant l'évolution du schéma SQL. La révision `20260901_01` crée le schéma `silver`, les quatre tables métier et les tables Ops utilisées pour tracer les runs et les anomalies. La révision `20260901_02` ajoute ensuite `silver.incident.comment` afin de préserver l'information métier sur les arrêts de machine.

La commande réellement utilisée dans le terminal était `uv run alembic upgrade head`. La cellule suivante ne rejoue pas la migration : elle vérifie le fichier versionné et lit la révision effectivement enregistrée dans PostgreSQL.

In [5]:
from sqlalchemy import text

from indusense.db.engine import create_database_engine

migration_path = PROJECT_ROOT / "migrations" / "versions" / "20260901_02_add_incident_comment.py"
migration_text = migration_path.read_text(encoding="utf-8")
assert 'revision: str = "20260901_02"' in migration_text
assert 'down_revision: str | Sequence[str] | None = "20260901_01"' in migration_text
assert 'sa.Column("comment", sa.Text(), nullable=True)' in migration_text

engine = create_database_engine()
with engine.connect() as connection:
    active_revision = connection.scalar(text("SELECT version_num FROM alembic_version"))

print(f"Migration : {migration_path.name}")
print(f"Révision PostgreSQL active : {active_revision}")
assert active_revision == "20260901_02"
print("Les migrations Silver et commentaire sont appliquées.")

Migration : 20260901_02_add_incident_comment.py
Révision PostgreSQL active : 20260901_02
Les migrations Silver et commentaire sont appliquées.


## 5. Publier Silver de façon atomique et traçable

La commande `uv run indusense build-silver` sélectionne les derniers lots Bronze terminés, transforme les quatre grains en mémoire, puis remplace toutes les tables Silver dans une transaction unique. Une publication **atomique** réussit entièrement ou ne laisse aucune table partiellement remplacée.

Chaque exécution crée un `pipeline_run` et l'associe aux lots Bronze utilisés. La cellule suivante affiche la commande pour mémoire, mais ne l'exécute pas : elle relit le dernier run réellement persisté.

In [6]:
BUILD_COMMAND = "uv run indusense build-silver"

with engine.connect() as connection:
    latest_run = connection.execute(
        text(
            """
            SELECT run_id, status, pipeline_version, transformation_version,
                   started_at, finished_at, metrics, error_message
            FROM ops.pipeline_run
            ORDER BY started_at DESC
            LIMIT 1
            """
        )
    ).mappings().one()
    source_batch_count = connection.scalar(
        text("SELECT COUNT(*) FROM ops.pipeline_run_source WHERE run_id = :run_id"),
        {"run_id": latest_run["run_id"]},
    )

print(f"Commande de publication : {BUILD_COMMAND}")
print("Commande non relancée par ce notebook.")
print(f"Dernier run : {latest_run['run_id']}")
print(f"Statut : {latest_run['status']}")
print(f"Lots Bronze liés : {source_batch_count}")
print(f"Début : {latest_run['started_at']}")
print(f"Fin : {latest_run['finished_at']}")
assert latest_run["status"] == "completed"
assert latest_run["error_message"] is None
assert source_batch_count == 3

Commande de publication : uv run indusense build-silver
Commande non relancée par ce notebook.
Dernier run : 2ce5c088-3320-4d0f-be1e-a2a3e3ae4205
Statut : completed
Lots Bronze liés : 3
Début : 2026-09-01 16:49:48.329697+00:00
Fin : 2026-09-01 16:49:50.656375+00:00


## 6. Contrôler les volumes et l'audit

Un contrôle de volume rapproche le nombre de lignes lues, publiées et écartées. Ici, `lignes lues = lignes publiées + doublons télémétriques écartés`. Les lignes conservées avec un capteur manquant restent comptées parmi les lignes publiées et possèdent un avertissement d'audit.

La cellule suivante compare les métriques du run avec les volumes réellement présents dans les tables Silver et détaille les avertissements.

In [7]:
expected_counts = {
    "machine": 15,
    "incident": 1_245,
    "maintenance": 1_562,
    "telemetry": 134_280,
}

with engine.connect() as connection:
    observed_counts = {
        table_name: connection.scalar(text(f"SELECT COUNT(*) FROM silver.{table_name}"))
        for table_name in expected_counts
    }
    issue_rows = connection.execute(
        text(
            """
            SELECT rule_code, severity, action, COUNT(*) AS occurrences
            FROM ops.transformation_issue
            WHERE run_id = :run_id
            GROUP BY rule_code, severity, action
            ORDER BY rule_code
            """
        ),
        {"run_id": latest_run["run_id"]},
    ).mappings().all()

volume_table = [
    [table_name, expected_counts[table_name], observed_counts[table_name]]
    for table_name in expected_counts
]
print(tabulate(volume_table, headers=["Table Silver", "Attendu", "Observé"], tablefmt="github"))
print("\nAudit du run :")
print(tabulate(issue_rows, headers="keys", tablefmt="github"))

metrics = latest_run["metrics"]
assert observed_counts == expected_counts
assert metrics["rows_read"] == metrics["rows_written"] + metrics["telemetry_duplicates_removed"]
assert sum(row["occurrences"] for row in issue_rows) == metrics["issues"]
print("\nVolumes et audit cohérents.")

| Table Silver   |   Attendu |   Observé |
|----------------|-----------|-----------|
| machine        |        15 |        15 |
| incident       |      1245 |      1245 |
| maintenance    |      1562 |      1562 |
| telemetry      |    134280 |    134280 |

Audit du run :
| rule_code                     | severity   | action             |   occurrences |
|-------------------------------|------------|--------------------|---------------|
| TELEMETRY_BUS_DUPLICATE       | warning    | deduplicated       |          1346 |
| TELEMETRY_MISSING_MEASUREMENT | warning    | retained_with_null |          2792 |

Volumes et audit cohérents.


## 7. Vérifier l'intégrité métier après publication

Les derniers contrôles recherchent des références orphelines, une maintenance encore associée à une autre machine que son incident, une clé télémétrique dupliquée, une colonne opérateur minimisée encore présente ou un commentaire absent. Une valeur nulle dans les résultats suivants signifierait que le contrôle lui-même est incomplet ; la valeur attendue est explicitement zéro échec.

La cellule affiche aussi quelques maintenances réellement réalignées afin de rendre la transformation observable.

In [8]:
integrity_queries = {
    "incidents sans machine": """
        SELECT COUNT(*) FROM silver.incident i
        LEFT JOIN silver.machine m ON m.machine_code = i.machine_code
        WHERE m.machine_code IS NULL
    """,
    "maintenances sans machine": """
        SELECT COUNT(*) FROM silver.maintenance x
        LEFT JOIN silver.machine m ON m.machine_code = x.machine_code
        WHERE m.machine_code IS NULL
    """,
    "télémétries sans machine": """
        SELECT COUNT(*) FROM silver.telemetry t
        LEFT JOIN silver.machine m ON m.machine_code = t.machine_code
        WHERE m.machine_code IS NULL
    """,
    "divergences maintenance-incident": """
        SELECT COUNT(*) FROM silver.maintenance x
        JOIN silver.incident i ON i.incident_id = x.related_incident_id
        WHERE x.machine_code <> i.machine_code
    """,
    "clés télémétriques dupliquées": """
        SELECT COUNT(*) FROM (
            SELECT machine_code, measured_at
            FROM silver.telemetry
            GROUP BY machine_code, measured_at
            HAVING COUNT(*) > 1
        ) duplicates
    """,
    "colonnes opérateur minimisées restantes": """
        SELECT COUNT(*) FROM information_schema.columns
        WHERE table_schema = 'silver' AND table_name = 'incident'
          AND column_name IN ('operator_name', 'operator_badge', 'shift')
    """,
    "commentaires absents": """
        SELECT COUNT(*) FROM silver.incident
        WHERE comment IS NULL
    """,
}

with engine.connect() as connection:
    integrity_results = [
        [check_name, connection.scalar(text(query))]
        for check_name, query in integrity_queries.items()
    ]
    aligned_examples = connection.execute(
        text(
            """
            SELECT maintenance_id, source_machine_code, machine_code, related_incident_id
            FROM silver.maintenance
            WHERE machine_code_was_aligned
            ORDER BY maintenance_id
            LIMIT 5
            """
        )
    ).mappings().all()

print(tabulate(integrity_results, headers=["Contrôle", "Échecs"], tablefmt="github"))
assert all(failure_count == 0 for _, failure_count in integrity_results)
print("\nExemples de maintenances réalignées :")
print(tabulate(aligned_examples, headers="keys", tablefmt="github"))

engine.dispose()
print("\nIntégrité Silver validée ; connexion libérée.")

| Contrôle                                |   Échecs |
|-----------------------------------------|----------|
| incidents sans machine                  |        0 |
| maintenances sans machine               |        0 |
| télémétries sans machine                |        0 |
| divergences maintenance-incident        |        0 |
| clés télémétriques dupliquées           |        0 |
| colonnes opérateur minimisées restantes |        0 |
| commentaires absents                    |        0 |

Exemples de maintenances réalignées :
|   maintenance_id | source_machine_code   | machine_code   | related_incident_id   |
|------------------|-----------------------|----------------|-----------------------|
|               91 | MACH-05               | MACH-01        | INC-000037            |
|               92 | MACH-01               | MACH-04        | INC-000042            |
|               93 | MACH-15               | MACH-06        | INC-000068            |
|               94 | MACH-10        

## Synthèse à savoir expliquer

- **Séparation des grains :** Silver possède une table par entité métier, contrairement au stockage source fidèle de Bronze.
- **Typage :** les textes Bronze deviennent des dates UTC, booléens, entiers, décimaux et mesures numériques contrôlées.
- **Relations :** les clés étrangères empêchent les machines inconnues et garantissent la cohérence entre maintenance réactive et incident.
- **Réalignement :** la machine de l'incident est la référence pour chaque maintenance réactive liée ; la valeur source reste traçable.
- **Dédoublonnage :** une seule télémétrie est publiée par machine et timestamp, sans moyenner arbitrairement les mesures.
- **Commentaires métier :** `comment` est conservé dans Silver, car il peut signaler un arrêt de machine absent des indicateurs structurés ; les informations opérateur inutiles restent exclues.
- **Traçabilité :** chaque ligne Silver conserve son lot, sa ligne, son empreinte et le run de transformation ; les écarts sont audités.
- **Atomicité :** le remplacement complet est validé dans une seule transaction ou annulé en cas d'erreur bloquante.

**Validation réellement observée :** après exécution du notebook, la révision `20260901_02` est appliquée, le dernier run est terminé sans erreur, 137 102 lignes métier sont publiées depuis 138 448 lignes Bronze, les 1 245 commentaires sont présents et tous les contrôles d'intégrité retournent zéro échec.